In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
import joblib
import matplotlib.pyplot as plt

In [ ]:
# ==========================================
# 1. Read and preprocess the data
# ==========================================
print("Reading data...")
# Assuming the file has a 'datetime' column for dates and a 'close' column for prices
try:
    df = pd.read_csv('../data/BTCUSD1D.csv')
except FileNotFoundError:
    print("Error: File 'BTCUSD1D.csv' not found.")
    exit()

In [ ]:
# Convert datetime column to datetime format and sort chronologically
df['datetime'] = pd.to_datetime(df['datetime'])
df = df.sort_values('datetime')
df = df.set_index('datetime')

In [ ]:
# ==========================================
# 2. Create Features (X) and Target (y)
# ==========================================
# Use the close prices of 30 consecutive sessions (t-1, t-2, ..., t-30) to predict session t
window_size = 30

In [ ]:
print(f"Creating data with a window of {window_size} sessions...")
for i in range(1, window_size + 1):
    df[f'Lag_{i}'] = df['close'].shift(i)

In [ ]:
# Drop the first 30 rows with missing data (NaN) caused by the shift operation
df = df.dropna()

In [ ]:
# List of feature columns (from Lag_1 to Lag_30)
feature_cols = [f'Lag_{i}' for i in range(1, window_size + 1)]

In [ ]:
X = df[feature_cols]
y = df['close']

In [ ]:
# ==========================================
# 3. Split Train / Test sets
# ==========================================
split_date = '2026-03-01'

In [ ]:
X_train = X.loc[X.index < split_date]
y_train = y.loc[y.index < split_date]

In [ ]:
X_test = X.loc[X.index >= split_date]
y_test = y.loc[y.index >= split_date]

In [ ]:
print(f"Number of training samples: {len(X_train)}")
print(f"Number of test samples: {len(X_test)}")

In [ ]:
if len(X_train) == 0 or len(X_test) == 0:
    print("Error: Insufficient data for training or testing based on the split date.")
    exit()

In [ ]:
# ==========================================
# 4. Train the Linear Regression model
# ==========================================
print("Training model...")
model = LinearRegression()
model.fit(X_train, y_train)

In [ ]:
# Basic model evaluation
train_score = model.score(X_train, y_train)
test_score = model.score(X_test, y_test)
print(f"R^2 training accuracy: {train_score:.4f}")
print(f"R^2 test accuracy:  {test_score:.4f}")

In [ ]:
# Calculate the error on the test set
y_pred = model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
print(f"Mean Absolute Error (MAE) on test set: {mae:.2f}")
print(f"Mean Squared Error (MSE) on test set: {mse:.2f}")

In [ ]:
# ==========================================
# 5. Save the model for reuse
# ==========================================
model_filename = 'models/1_lr_btc_30days_model.pkl'
joblib.dump(model, model_filename)
print(f"\nModel successfully saved to file: '{model_filename}'")

In [ ]:
# ==========================================
# 6. Display Predicted vs Actual Results
# ==========================================
print("\nPredictions from 2026-03-01 onwards:")
results_df = pd.DataFrame({'Actual Price': y_test, 'Predicted Price': y_pred})
print(results_df.head(20)) # Display the first 20 rows of predictions and actual values

In [ ]:
# ==========================================
# 7. Visualize predicted and actual prices
# ==========================================
plt.figure(figsize=(12, 6))
plt.plot(results_df.index, results_df['Actual Price'], label='Actual Price', color='blue', alpha=0.7)
plt.plot(results_df.index, results_df['Predicted Price'], label='Predicted Price', color='red', linestyle='--', alpha=0.7)
plt.title('Bitcoin (BTC) Price Prediction - Linear Regression')
plt.xlabel('Date')
plt.ylabel('Close Price (USD)')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# ==========================================
# 8. Example: Predict the next session's price
# (Using the current most recent 30 sessions from the original data)
# ==========================================
if not df.empty:
    # Get the 30 most recent values from the original data, reverse them to match the t-1 to t-30 order
    recent_30_days_features = df['close'].iloc[-window_size:].values[::-1].reshape(1, -1)

    # Load the model (simulating usage in another script/environment)
    loaded_model = joblib.load(model_filename)

    # Note: this is a simple prediction on the latest available data, not continuing the sequence of predictions
    next_session_pred = loaded_model.predict(recent_30_days_features)
    print(f"\n--> Predicted next session's BTC price based on the very last available 30 days is: {next_session_pred[0]:.2f}")
else:
    print("\nError: Dataframe is empty, cannot make the single next session prediction.")